# Projet : ChestScan AI - Classification de Radiographies Thoraciques (Deep Learning)

In [16]:

!pip install pandas
!pip install seaborn
!pip install scikit-learn
!pip install kaggle

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 181.2/181.2 KB 827.8 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.2/78.2 KB 406.2 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 165.1/165.1 KB 707.2 kB/s eta 0:00:00a 0:00:01


In [10]:

import os
import subprocess
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn import metrics
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import Sequential, Model
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.losses import BinaryCrossentropy
from tensorflow.keras.optimizers import SGD
from tensorflow.keras.metrics import Precision,Recall,F1Score,AUC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve


2026-10-23 15:40:27.733134: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1792770027.779177    1661 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1792770027.800442    1661 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1792770027.870117    1661 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1792770027.870129    1661 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1792770027.870132    1661 computation_placer.cc:177] computation placer alr

In [14]:
PROJECT_DIR=Path("/content/LungsLensAi")
SCRIPT_DIR= PROJECT_DIR / "scripts"
KAGGLE_DATASET="paultimothymooney/chest-xray-pneumonia"

DATASET_ROOT= PROJECT_DIR / "data"
TRAIN_DIR = DATASET_ROOT / "chest_xray" / "train"
VAL_DIR = DATASET_ROOT / "chest_xray" / "val"
TEST_DIR = DATASET_ROOT / "chest_xray" / "test"

SEED = 42
IMG_SIZE = (180, 180)
IMG_SIZE_MOBINET = (224, 180)
BATCH_SIZE = 32
EPOCHS = 20
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]

OUTPUT_DIR = PROJECT_DIR / "output"
FINAL_MODEL_PATH = OUTPUT_DIR / "model_final.keras"
CONFIG_PATH = OUTPUT_DIR / "model_config.json"
IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg"
}
CHECKPOINT_PATH= OUTPUT_DIR / "best_model.keras"

## 1. CHARGEMENT ET EDA

### 1. Chargement des donnees

In [15]:
def download_dataset(dataset_path, kaggle_dataset):
    if os.path.exists(dataset_path) and os.listdir(dataset_path):
        print("Dataset already exists. Skipping download.")
    else:
        print("Downloading dataset from Kaggle...")
        os.makedirs(dataset_path, exist_ok=True)

        subprocess.run(
            [
                "kaggle", "datasets", "download",
                "-d", kaggle_dataset,
                "-p", dataset_path,
                "--unzip"
            ],
            check=True
        ) 
download_dataset(DATASET_ROOT,KAGGLE_DATASET)

FileNotFoundError: [Errno 2] No such file or directory: 'kaggle'

### 1. Images per class

In [ ]:
def count_images(folder):
    return sum(
        1 for path in folder.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )
for split in ["train", "val", "test"]:
    split_dir = DATASET_ROOT / split

    if split_dir.exists():
        print(f"\n{split.upper()}")

        for class_name in ["NORMAL", "PNEUMONIA"]:
            class_dir = split_dir / class_name

            count = count_images(class_dir) if class_dir.exists() else 0
            print(f"  {class_name}: {count} images")

### 2. Distribution des classes

In [ ]:
distribution = []

for split in ["train", "val", "test"]:
    split_dir = DATASET_ROOT / "chest_xray" / "chest_xray" / split

    if split_dir.exists():
        for class_name in ["NORMAL", "PNEUMONIA"]:
            class_dir = split_dir / class_name

            distribution.append({
                "Split": split,
                "Classe": class_name,
                "Nombre": (
                    count_images(class_dir)
                    if class_dir.exists() else 0
                )
            })

df_distribution = pd.DataFrame(distribution)
display(df_distribution)

plt.figure(figsize=(9, 5))
sns.barplot(
    data=df_distribution,
    x="Split",
    y="Nombre",
    hue="Classe"
)
plt.title("Distribution des radiographies par classe")
plt.ylabel("Nombre d'images")
plt.tight_layout()
plt.show()

## 2. CHARGEMENT ET PRÉTRAITEMENT DU DATASET

In [ ]:
# Datasets pour le CNN from scratch (Images 180x180)
train_ds_180 = image_dataset_from_directory(TRAIN_DIR, image_size=(180, 180), batch_size=BATCH_SIZE, seed=SEED, label_mode='binary')
val_ds_180 = image_dataset_from_directory(VAL_DIR, image_size=(180, 180), batch_size=BATCH_SIZE, seed=SEED, label_mode='binary')
test_ds_180 = image_dataset_from_directory(TEST_DIR, image_size=(180, 180), batch_size=BATCH_SIZE, seed=SEED, label_mode='binary', shuffle=False)

# Datasets pour MobileNetV2 (Images 224x224)
train_ds_224 = image_dataset_from_directory(TRAIN_DIR, image_size=(224, 224), batch_size=BATCH_SIZE, seed=SEED, label_mode='binary')
val_ds_224 = image_dataset_from_directory(VAL_DIR, image_size=(224, 224), batch_size=BATCH_SIZE, seed=SEED, label_mode='binary')
test_ds_224 = image_dataset_from_directory(TEST_DIR, image_size=(224, 224), batch_size=BATCH_SIZE, seed=SEED, label_mode='binary', shuffle=False)

# Prétraitement et optimisation des performances
normalization_layer = tf.keras.layers.Rescaling(1./255)
train_ds_180 = train_ds_180.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds_180 = val_ds_180.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds_180 = test_ds_180.map(lambda x, y: (normalization_layer(x), y)).cache().prefetch(buffer_size=tf.data.AUTOTUNE)

mobile_preprocess = tf.keras.applications.mobilenet_v2.preprocess_input
train_ds_224 = train_ds_224.map(lambda x, y: (mobile_preprocess(x), y)).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
val_ds_224 = val_ds_224.map(lambda x, y: (mobile_preprocess(x), y)).cache().prefetch(buffer_size=tf.data.AUTOTUNE)
test_ds_224 = test_ds_224.map(lambda x, y: (mobile_preprocess(x), y)).cache().prefetch(buffer_size=tf.data.AUTOTUNE)


## 3. CONCEPTION ET ENTRAÎNEMENT DU CNN CUSTOM

In [ ]:
callbacks = [
    ModelCheckpoint(filepath=str(CHECKPOINT_PATH),monitor="val_auc",mode="max",save_best_only=True,verbose=1),
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=2, min_lr=1e-6)
]

data_augmentation = Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.05),
])

cnn_model = Sequential([
    data_augmentation,
    Conv2D(32, (3, 3), activation='relu', input_shape=(180, 180, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

cnn_model.compile(optimizer=SGD(), loss='binary_crossentropy', metrics=[Precision(),Recall(),F1Score(),AUC()])
history_cnn = cnn_model.fit(train_ds_180, validation_data=val_ds_180, epochs=EPOCHS, callbacks=callbacks)


In [ ]:
X_test = np.concatenate([
    images.numpy()
    for images, labels in test_ds_180
], axis=0)

y_test = np.concatenate([
    labels.numpy()
    for images, labels in test_ds_180
], axis=0)
y_pred = (cnn_model.predict(X_test) > 0.5).astype(int)
confusion = metrics.confusion_matrix(y_true=y_test ,y_pred=y_pred)

metrics.ConfusionMatrixDisplay.from_predictions(y_test,y_pred,display_labels=["NORMAL","PNEUMONIA"])
plt.show()


## 5. TRANSFER LEARNING AVEC MOBILENETV2

In [ ]:
base_model = MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = GlobalAveragePooling2D()(x)
x = Dropout(0.3)(x)
outputs = Dense(1, activation='sigmoid')(x)

mobilenet_model = Model(inputs, outputs)
mobilenet_model.compile(optimizer=SGD(), loss=BinaryCrossentropy(),metrics=[Precision(),Recall(),F1Score(),AUC()])

history_mb = mobilenet_model.fit(train_ds_224, validation_data=val_ds_224, epochs=EPOCHS, callbacks=callbacks)


## 6. FINE-TUNING DE MOBILENETV2

In [ ]:
base_model.trainable = True
fine_tune_at = 100
for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

mobilenet_model.compile(
    optimizer=SGD(learning_rate=1e-5),
    loss=BinaryCrossentropy(),
    metrics=[Precision(),Recall(),F1Score(),AUC()]
)

history_fine = mobilenet_model.fit(train_ds_224, validation_data=val_ds_224, epochs=5, callbacks=callbacks)


In [ ]:
# y_pred = (mobilenet_model.predict(X_test) > 0.5).astype(int)
# confusion = metrics.confusion_matrix(y_true=y_test ,y_pred=y_pred)

# metrics.ConfusionMatrixDisplay.from_predictions(y_test,y_pred,display_labels=["NORMAL","PNEUMONIA"])
# plt.show()
mobilenet_model.predict(X_test)

## 7. ÉVALUATION ET COMPARAISON

In [ ]:
def evaluate_model(model, test_dataset, name):
    y_true = np.concatenate([y for x, y in test_dataset], axis=0)
    y_pred_probs = model.predict(test_dataset)
    y_pred = (y_pred_probs >= 0.5).astype(int)
    auc = roc_auc_score(y_true, y_pred_probs)
    print(f"[{name}] ROC-AUC : {auc:.4f}")
    return y_true, y_pred_probs

_, _ = evaluate_model(cnn_model, test_ds_180, "CNN Custom")
_, _ = evaluate_model(mobilenet_model, test_ds_224, "MobileNetV2 Fine-Tuned")

## 8. SAUVEGARDE DU MODÈLE RETENU ET FONCTION D'INFÉRENCE

In [ ]:
mobilenet_model.save(FINAL_MODEL_PATH)
print(f"Modèle final sauvegardé sous : {FINAL_MODEL_PATH}")